In [200]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

Ссылка на датасет: [Classification of oil and gas](https://www.kaggle.com/competitions/classification-of-oil-and-gas)

In [201]:
train_df = pd.read_csv("train_oil.csv")
test_df = pd.read_csv("oil_test.csv")

In [202]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Field name                      309 non-null    object 
 1   Reservoir unit                  309 non-null    object 
 2   Country                         282 non-null    object 
 3   Region                          271 non-null    object 
 4   Basin name                      271 non-null    object 
 5   Tectonic regime                 309 non-null    object 
 6   Latitude                        282 non-null    float64
 7   Longitude                       279 non-null    float64
 8   Operator company                309 non-null    object 
 9   Onshore/Offshore                309 non-null    object 
 10  Hydrocarbon type                309 non-null    object 
 11  Reservoir status                309 non-null    object 
 12  Structural setting              309 

In [203]:
train_df.head()

,Field name,Reservoir unit,Country,Region,Basin name,Tectonic regime,Latitude,Longitude,Operator company,Onshore/Offshore,Hydrocarbon type,Reservoir status,Structural setting,Depth,Reservoir period,Lithology,Thickness (gross average ft),Thickness (net pay average ft),Porosity,Permeability
0,ZHIRNOV,MELEKESKIAN,RUSSIA,FORMER SOVIET UNION,VOLGA-URAL,COMPRESSION/EVAPORITE,51.0000,44.8042,NIZHNEVOLZHSKNET,ONSHORE,OIL,DECLINING PRODUCTION,FORELAND,1870,CARBONIFEROUS,SANDSTONE,262.0,33.0,24.0,30.0
1,LAGOA PARDA,LAGOA PARDA (URUCUTUCA),BRAZIL,LATIN AMERICA,ESPIRITO SANTO,EXTENSION,-19.6017,-39.8332,PETROBRAS,ONSHORE,OIL,NEARLY DEPLETED,PASSIVE MARGIN,4843,PALEOGENE,SANDSTONE,2133.0,72.0,23.0,350.0
2,ABQAIQ,ARAB D,SAUDI ARABIA,MIDDLE EAST,THE GULF,COMPRESSION/EVAPORITE,26.0800,49.8100,SAUDI ARAMCO,ONSHORE,OIL,REJUVENATING,FORELAND,6050,JURASSIC,LIMESTONE,250.0,184.0,21.0,410.0
3,MURCHISON,BRENT,UK /NORWAY,EUROPE,NORTH SEA NORTHERN,EXTENSION,61.3833,1.7500,CNR,OFFSHORE,OIL,NEARLY DEPLETED,RIFT,8988,JURASSIC,SANDSTONE,425.0,300.0,22.0,750.0
4,WEST PEMBINA,NISKU (PEMBINA L POOL),CANADA,NORTH AMERICA,WESTERN CANADA,COMPRESSION,53.2287,-115.8008,NUMEROUS,ONSHORE,OIL,UNKNOWN,FORELAND,9306,DEVONIAN,DOLOMITE,233.0,167.0,11.8,1407.0


In [204]:
print(train_df["Field name"].value_counts())
print("="*50)
print("Это просто название месторождения которое совсем "
      "не повторяется и невозможно достать какую-либо "
      "закономерность (почти как ID) => дропаем")

train_df = train_df.drop("Field name", axis=1)

Field name
ERSKINE                        3
LAOJUNMIAO                     3
ZAKUM                          3
KHAFJI                         2
ELK BASIN                      2
                              ..
PEEJAY                         1
TABER NORTH                    1
WHITNEY CANYON-CARTER CREEK    1
NORTH ROBERTSON                1
NORTH SABINE LAKE              1
Name: count, Length: 285, dtype: int64
Это просто название месторождения которое совсем не повторяется и невозможно достать какую-либо закономерность (почти как ID) => дропаем


In [205]:
print(train_df["Reservoir unit"].value_counts())
print("="*50)
print("Практически аналогичная история")

train_df = train_df.drop("Reservoir unit", axis=1)

Reservoir unit
BRENT                               8
SAN ANDRES                          7
SHUAIBA                             5
LEMAN SANDSTONE                     4
TOR-EKOFISK                         4
                                   ..
ASMARI                              1
WATT MOUNTAIN (GILWOOD A)           1
LOWER GANCHAIGOU                    1
ZELTEN                              1
MERECURE (NARICUAL-LOS JABILLOS)    1
Name: count, Length: 258, dtype: int64
Практически аналогичная история


In [206]:
train_df[["Country"]]

,Country
0,RUSSIA
1,BRAZIL
2,SAUDI ARABIA
3,UK /NORWAY
4,CANADA
...,...
304,PAPUA NEW GUINEA
305,CANADA
306,USA
307,USA


In [207]:
train_df["Country_split"] = train_df["Country"].str.split(" /")
train_df["Country_split"] = train_df["Country_split"].apply(
    lambda x: x if isinstance(x, list) else ["MISSING"]
)

all_country_series = train_df["Country_split"].explode()
top7_country = all_country_series.value_counts().head(7).index

train_df["Country_top7"] = train_df["Country_split"].apply(
    lambda lst: [m for m in lst if m in top7_country]
)

mlb_Country = MultiLabelBinarizer()
country_encoded = mlb_Country.fit_transform(train_df["Country_top7"])

country_df = pd.DataFrame(
    country_encoded,
    columns=[f"COUNTRY_{cls}" for cls in mlb_Country.classes_]
)

train_df = pd.concat([train_df, country_df], axis=1)
train_df = train_df.drop(columns=["Country", "Country_split", "Country_top7"])

In [208]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Region                          271 non-null    object 
 1   Basin name                      271 non-null    object 
 2   Tectonic regime                 309 non-null    object 
 3   Latitude                        282 non-null    float64
 4   Longitude                       279 non-null    float64
 5   Operator company                309 non-null    object 
 6   Onshore/Offshore                309 non-null    object 
 7   Hydrocarbon type                309 non-null    object 
 8   Reservoir status                309 non-null    object 
 9   Structural setting              309 non-null    object 
 10  Depth                           309 non-null    int64  
 11  Reservoir period                309 non-null    object 
 12  Lithology                       309 

In [209]:
train_df["Region"].unique()

array(['FORMER SOVIET UNION', 'LATIN AMERICA', 'MIDDLE EAST', 'EUROPE',
       'NORTH AMERICA', nan, 'FAR EAST', 'AFRICA'], dtype=object)

In [210]:
train_df["Region"] = train_df["Region"].fillna("MISSING")

ohe_region = OneHotEncoder(drop="first", sparse_output=False)
region_encoded = ohe_region.fit_transform(train_df[["Region"]])

region_cols = ohe_region.get_feature_names_out(["Region"])
region_encoded_df = pd.DataFrame(region_encoded, columns=region_cols, index=train_df.index)

train_df = pd.concat([train_df, region_encoded_df], axis=1)

train_df = train_df.drop("Region", axis=1)

In [211]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 30 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Basin name                      271 non-null    object 
 1   Tectonic regime                 309 non-null    object 
 2   Latitude                        282 non-null    float64
 3   Longitude                       279 non-null    float64
 4   Operator company                309 non-null    object 
 5   Onshore/Offshore                309 non-null    object 
 6   Hydrocarbon type                309 non-null    object 
 7   Reservoir status                309 non-null    object 
 8   Structural setting              309 non-null    object 
 9   Depth                           309 non-null    int64  
 10  Reservoir period                309 non-null    object 
 11  Lithology                       309 non-null    object 
 12  Thickness (gross average ft)    309 

In [212]:
train_df["Basin name"].value_counts()

Basin name
WESTERN CANADA                     24
GULF OF MEXICO NORTHERN ONSHORE    19
NORTH SEA CENTRAL                  14
NORTH SEA NORTHERN                 12
WILLISTON                          10
                                   ..
JUNGGAR (ZHUNGEER)                  1
TARAKAN                             1
NORTH SAKHALIN                      1
SVERDRUP                            1
UINTA                               1
Name: count, Length: 93, dtype: int64

In [213]:
train_df["Basin_name_split"] = train_df["Basin name"].str.split("/")
train_df["Basin_name_split"] = train_df["Basin_name_split"].apply(
    lambda x: x if isinstance(x, list) else ["MISSING"]
)

all_basin_series = train_df["Basin_name_split"].explode()
top8_basin = all_basin_series.value_counts().head(8).index

train_df["Basin_top8"] = train_df["Basin_name_split"].apply(
    lambda lst: [m for m in lst if m in top8_basin]
)

mlb_Basin = MultiLabelBinarizer()
basin_encoded = mlb_Basin.fit_transform(train_df["Basin_top8"])

basin_df = pd.DataFrame(
    basin_encoded,
    columns=[f"BASIN_{cls}" for cls in mlb_Basin.classes_]
)

train_df = pd.concat([train_df, basin_df], axis=1)
train_df = train_df.drop(columns=["Basin name", "Basin_name_split", "Basin_top8"])

In [214]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 37 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Tectonic regime                        309 non-null    object 
 1   Latitude                               282 non-null    float64
 2   Longitude                              279 non-null    float64
 3   Operator company                       309 non-null    object 
 4   Onshore/Offshore                       309 non-null    object 
 5   Hydrocarbon type                       309 non-null    object 
 6   Reservoir status                       309 non-null    object 
 7   Structural setting                     309 non-null    object 
 8   Depth                                  309 non-null    int64  
 9   Reservoir period                       309 non-null    object 
 10  Lithology                              309 non-null    object 
 11  Thickn

In [215]:
train_df["Tectonic regime"].unique()

array(['COMPRESSION/EVAPORITE', 'EXTENSION', 'COMPRESSION',
       'INVERSION/COMPRESSION/EXTENSION/EVAPORITE',
       'GRAVITY/EXTENSION/SHALE/SYNSEDIMENTATION', 'COMPRESSION/EROSION',
       'EXTENSION/EROSION', 'EXTENSION/EVAPORITE', 'COMPRESSION/SHALE',
       'GRAVITY/EVAPORITE/EXTENSION',
       'STRIKE-SLIP/INVERSION/COMPRESSION/EXTENSION',
       'INVERSION/COMPRESSION/EXTENSION', 'GRAVITY/SHALE/EXTENSION',
       'TRANSTENSION/EXTENSION/SHALE/LINKED',
       'GRAVITY/EVAPORITE/COMPRESSION',
       'INVERSION/STRIKE-SLIP/TRANSPRESSION/EXTENSION/BASEMENT-I',
       'INVERSION/COMPRESSION/EXTENSION/EROSION',
       'GRAVITY/EXTENSION/EVAPORITE/SYNSEDIMENTATION',
       'COMPRESSION/EXTENSION/LINKED', 'EXTENSION/INVERSION',
       'COMPRESSION/EXTENSION/EVAPORITE',
       'STRIKE-SLIP/TRANSTENSION/BASEMENT-I',
       'GRAVITY/EXTENSION/EVAPORITE',
       'INVERSION/COMPRESSION/EXTENSION/EVAPORITE/GRAVITY',
       'EXTENSION/EVAPORITE/EROSION/GRAVITY', 'EXTENSION/TRANSTENSION',
   

In [216]:
train_df["Tectonic_regime_split"] = train_df["Tectonic regime"].str.split("/")

mlb_TecRegime = MultiLabelBinarizer()
tecRegime_encoded = mlb_TecRegime.fit_transform(train_df["Tectonic_regime_split"])

tecRegime_df = pd.DataFrame(
    tecRegime_encoded,
    columns=[f"TecRegime_{cls}" for cls in mlb_TecRegime.classes_]
)

train_df = pd.concat([train_df, tecRegime_df], axis=1)
train_df = train_df.drop(columns=["Tectonic regime", "Tectonic_regime_split"])

In [217]:
train_df[["Latitude", "Longitude"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Latitude   282 non-null    float64
 1   Longitude  279 non-null    float64
dtypes: float64(2)
memory usage: 5.0 KB


In [218]:
train_df["Latitude"] = train_df["Latitude"].fillna(train_df["Latitude"].median())
train_df["Longitude"] = train_df["Longitude"].fillna(train_df["Longitude"].median())

In [219]:
train_df["Operator_company_split"] = train_df["Operator company"].str.split(" /")

all_company_series = train_df["Operator_company_split"].explode()
top7_company = all_company_series.value_counts().head(7).index

train_df["Company_top7"] = train_df["Operator_company_split"].apply(
    lambda lst: [m for m in lst if m in top7_company]
)

mlb_Operator = MultiLabelBinarizer()
company_encoded = mlb_Operator.fit_transform(train_df["Company_top7"])

company_df = pd.DataFrame(
    company_encoded,
    columns=[f"COMPANY_{cls}" for cls in mlb_Operator.classes_]
)

train_df = pd.concat([train_df, company_df], axis=1)
train_df = train_df.drop(columns=["Operator company", "Operator_company_split", "Company_top7"])

In [220]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 58 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Latitude                               309 non-null    float64
 1   Longitude                              309 non-null    float64
 2   Onshore/Offshore                       309 non-null    object 
 3   Hydrocarbon type                       309 non-null    object 
 4   Reservoir status                       309 non-null    object 
 5   Structural setting                     309 non-null    object 
 6   Depth                                  309 non-null    int64  
 7   Reservoir period                       309 non-null    object 
 8   Lithology                              309 non-null    object 
 9   Thickness (gross average ft)           309 non-null    float64
 10  Thickness (net pay average ft)         309 non-null    float64
 11  Porosi

In [221]:
on_off = {
    "ONSHORE": 1,
    "OFFSHORE": 0,
    "ONSHORE-OFFSHORE": 2
}

train_df["Onshore/Offshore"] = train_df["Onshore/Offshore"].map(on_off)

In [222]:
ohe_type = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
type_encoded = ohe_type.fit_transform(train_df[["Hydrocarbon type"]])

type_cols = ohe_type.get_feature_names_out()
type_df = pd.DataFrame(
    type_encoded,
    columns=type_cols
)

train_df = pd.concat([train_df, type_df], axis=1)
train_df = train_df.drop(columns=["Hydrocarbon type"])


In [223]:
ohe_status = OneHotEncoder(drop="first", sparse_output=False)

status_encoded = ohe_status.fit_transform(train_df[["Reservoir status"]])
status_cols = ohe_status.get_feature_names_out()

status_df = pd.DataFrame(
    status_encoded,
    columns=status_cols
)

train_df = pd.concat([train_df, status_df], axis=1)
train_df = train_df.drop(columns=["Reservoir status"])

In [224]:
train_df["Structural_settings_split"] = train_df["Structural setting"].str.split("/")

mlb_settings = MultiLabelBinarizer()
settings_encoded = mlb_settings.fit_transform(train_df["Structural_settings_split"])

settings_df = pd.DataFrame(
    settings_encoded,
    columns=[f"SETTINGS_{cls}" for cls in mlb_settings.classes_]
)

train_df = pd.concat([train_df, settings_df], axis=1)
train_df = train_df.drop(columns=["Structural setting", "Structural_settings_split"])

In [225]:
top_periods = train_df["Reservoir period"].value_counts().head(9).index

train_df["Reservoir_period_top9"] = train_df["Reservoir period"].apply(
    lambda x: x if x in top_periods else "OTHER"
)

ohe_periods = OneHotEncoder(drop="first", sparse_output=False)
periods_encoded = ohe_periods.fit_transform(train_df[["Reservoir_period_top9"]])

periods_cols = ohe_periods.get_feature_names_out()

periods_df = pd.DataFrame(
    periods_encoded,
    columns=periods_cols
)

train_df = pd.concat([train_df, periods_df], axis=1)
train_df = train_df.drop(columns=["Reservoir period", "Reservoir_period_top9"])

In [226]:
top_lithology = train_df["Lithology"].value_counts().head(3).index

train_df["LithologyTOP"] = train_df["Lithology"].apply(
    lambda x: x if x in top_lithology else "OTHER"
)

ohe_lithology = OneHotEncoder(drop="first", sparse_output=False)
lithology_encoded = ohe_lithology.fit_transform(train_df[["LithologyTOP"]])

lithology_cols = ohe_lithology.get_feature_names_out()

lithology_df = pd.DataFrame(
    lithology_encoded,
    columns=lithology_cols
)

train_df = pd.concat([train_df, lithology_df], axis=1)
train_df = train_df.drop(columns=["Lithology", "LithologyTOP"])

Теперь то же самое с тестовыми данными

In [227]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Field name                      133 non-null    object 
 1   Reservoir unit                  133 non-null    object 
 2   Country                         120 non-null    object 
 3   Region                          117 non-null    object 
 4   Basin name                      125 non-null    object 
 5   Tectonic regime                 133 non-null    object 
 6   Latitude                        120 non-null    float64
 7   Longitude                       117 non-null    float64
 8   Operator company                133 non-null    object 
 9   Hydrocarbon type                133 non-null    object 
 10  Reservoir status                133 non-null    object 
 11  Structural setting              133 non-null    object 
 12  Depth                           133 

In [228]:
test_df = test_df.drop(["Field name", "Reservoir unit"], axis=1)

In [229]:
test_df["Country_split"] = test_df["Country"].str.split(" /")
test_df["Country_split"] = test_df["Country_split"].apply(
    lambda x: x if isinstance(x, list) else ["MISSING"]
)

test_df["Country_top7"] = test_df["Country_split"].apply(
    lambda lst: [m for m in lst if m in top7_country]
)

country_encoded_test = mlb_Country.transform(test_df["Country_top7"])
country_df_test = pd.DataFrame(
    country_encoded_test,
    columns=[f"COUNTRY_{cls}" for cls in mlb_Country.classes_]
)

test_df = pd.concat([test_df, country_df_test], axis=1)
test_df = test_df.drop(columns=["Country", "Country_split", "Country_top7"])

In [230]:
test_df["Region"] = test_df["Region"].fillna("MISSING")

region_encoded_test = ohe_region.transform(test_df[["Region"]])
region_cols = ohe_region.get_feature_names_out()

region_df_test = pd.DataFrame(
    region_encoded_test,
    columns=region_cols
)

test_df = pd.concat([test_df, region_df_test], axis=1)
test_df = test_df.drop(columns=["Region"])

In [231]:
test_df["Basin_name_split"] = test_df["Basin name"].str.split("/")
test_df["Basin_name_split"] = test_df["Basin_name_split"].apply(
    lambda x: x if isinstance(x, list) else ["MISSING"]
)

test_df["Basin_top8"] = test_df["Basin_name_split"].apply(
    lambda lst: [m for m in lst if m in top8_basin]
)

basin_encoded_test = mlb_Basin.transform(test_df["Basin_top8"])

basin_df_test = pd.DataFrame(
    basin_encoded_test,
    columns=[f"BASIN_{cls}" for cls in mlb_Basin.classes_]
)

test_df = pd.concat([test_df, basin_df_test], axis=1)
test_df = test_df.drop(columns=["Basin name", "Basin_name_split", "Basin_top8"])

In [232]:
test_df["Tectonic_regime_split"] = test_df["Tectonic regime"].str.split("/")

tecRegime_encoded_test = mlb_TecRegime.transform(test_df["Tectonic_regime_split"])

tecRegime_df_test = pd.DataFrame(
    tecRegime_encoded_test,
    columns=[f"TecRegime_{cls}" for cls in mlb_TecRegime.classes_]
)

test_df = pd.concat([test_df, tecRegime_df_test], axis=1)
test_df = test_df.drop(columns=["Tectonic regime", "Tectonic_regime_split"])

In [233]:
test_df["Latitude"] = test_df["Latitude"].fillna(train_df["Latitude"].median())
test_df["Longitude"] = test_df["Longitude"].fillna(train_df["Longitude"].median())

In [234]:
test_df["Operator_company_split"] = test_df["Operator company"].str.split(" /")

test_df["Company_top7"] = test_df["Operator_company_split"].apply(
    lambda lst: [m for m in lst if m in top7_company]
)

company_encoded_test = mlb_Operator.transform(test_df["Company_top7"])

company_df_test = pd.DataFrame(
    company_encoded_test,
    columns=[f"COMPANY_{cls}" for cls in mlb_Operator.classes_]
)

test_df = pd.concat([test_df, company_df_test], axis=1)
test_df = test_df.drop(columns=["Operator company", "Operator_company_split", "Company_top7"])

In [235]:
type_encoded_test = ohe_type.transform(test_df[["Hydrocarbon type"]])

type_df_test = pd.DataFrame(
    type_encoded_test,
    columns=type_cols
)

test_df = pd.concat([test_df, type_df_test], axis=1)
test_df = test_df.drop(columns=["Hydrocarbon type"])

In [237]:
status_encoded_test = ohe_status.transform(test_df[["Reservoir status"]])

status_df_test = pd.DataFrame(
    status_encoded_test,
    columns=status_cols
)

test_df = pd.concat([test_df, status_df_test], axis=1)
test_df = test_df.drop(columns=["Reservoir status"])

In [238]:
test_df["Structural_settings_split"] = test_df["Structural setting"].str.split("/")

settings_encoded_test = mlb_settings.transform(test_df["Structural_settings_split"])

settings_df_test = pd.DataFrame(
    settings_encoded_test,
    columns=[f"SETTINGS_{cls}" for cls in mlb_settings.classes_]
)

test_df = pd.concat([test_df, settings_df_test], axis=1)
test_df = test_df.drop(columns=["Structural setting", "Structural_settings_split"])

/home/zas020/Desktop/study/machine-learning/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) [''] will be ignored
  warnings.warn(


In [239]:
test_df["Reservoir_period_top9"] = test_df["Reservoir period"].apply(
    lambda x: x if x in top_periods else "OTHER"
)

periods_encoded_test = ohe_periods.transform(test_df[["Reservoir_period_top9"]])

periods_df_test = pd.DataFrame(
    periods_encoded_test,
    columns=periods_cols
)

test_df = pd.concat([test_df, periods_df_test], axis=1)
test_df = test_df.drop(columns=["Reservoir period", "Reservoir_period_top9"])

In [240]:
test_df["LithologyTOP"] = test_df["Lithology"].apply(
    lambda x: x if x in top_lithology else "OTHER"
)

lithology_encoded_test = ohe_lithology.transform(test_df[["LithologyTOP"]])

lithology_df_test = pd.DataFrame(
    lithology_encoded_test,
    columns=lithology_cols
)

test_df = pd.concat([test_df, lithology_df_test], axis=1)
test_df = test_df.drop(columns=["Lithology", "LithologyTOP"])

In [241]:
model = LogisticRegression(
    max_iter=1000,
    random_state=83,
    class_weight="balanced"
)

y_train = train_df["Onshore/Offshore"]
X_train = train_df.drop("Onshore/Offshore", axis=1)
X_test = test_df.copy()

In [242]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [243]:
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,83
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [244]:
predictions = model.predict(X_test_scaled)

sub_df = pd.DataFrame({
    "index": range(len(predictions)),
    "Onshore/Offshore": predictions
})

sub_df.to_csv("sub_df.csv", index=False)

In [245]:
sub_df.head()

,index,Onshore/Offshore
0,0,1
1,1,0
2,2,1
3,3,0
4,4,1
